In [ ]:
import os
import json
from datetime import datetime, timezone
from typing import List, Dict

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# =========================
# CONFIG
# =========================
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

DB_DIR = "data/news_chroma_db"


# =========================
# 1. INGESTION
# =========================
def load_news_json(filepath: str) -> List[Dict]:
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


# =========================
# 2. PREPROCESS + TAG EXTRACTION
# =========================
TECH_KEYWORDS = ["ai", "vr", "ar", "ray tracing", "cloud gaming"]
MONETIZATION_KEYWORDS = ["battle pass", "microtransaction", "loot box", "dlc"]
PSYCHOLOGICAL_KEYWORDS = ["stress", "emotion", "anxiety", "immersion"]
BEHAVIORAL_KEYWORDS = ["habit", "engagement", "addiction", "behavior"]

GAME_HINTS = ["fortnite", "minecraft", "roblox", "elden ring"]
COMPANY_HINTS = ["sony", "microsoft", "nintendo", "ubisoft", "ea"]
PLATFORM_HINTS = ["pc", "xbox", "playstation", "ps5", "switch", "mobile"]


def find_matches(text: str, vocab: List[str]) -> List[str]:
    text = text.lower()
    return list(set([v for v in vocab if v in text]))


def detect_event_type(text: str) -> str:
    text = text.lower()
    if any(k in text for k in MONETIZATION_KEYWORDS):
        return "monetization_change"
    if any(k in text for k in TECH_KEYWORDS):
        return "technology_update"
    if any(k in text for k in ["launch", "release", "update", "patch"]):
        return "release_update"
    return "general_news"


def preprocess_news(item: Dict) -> Dict:
    text = f"{item.get('title', '')} {item.get('summary', '')}"

    item["game_titles"] = find_matches(text, GAME_HINTS)
    item["companies"] = find_matches(text, COMPANY_HINTS)
    item["platforms"] = find_matches(text, PLATFORM_HINTS)

    item["technology_tags"] = find_matches(text, TECH_KEYWORDS)
    item["monetization_tags"] = find_matches(text, MONETIZATION_KEYWORDS)
    item["psychological_tags"] = find_matches(text, PSYCHOLOGICAL_KEYWORDS)
    item["behavioral_tags"] = find_matches(text, BEHAVIORAL_KEYWORDS)

    item["event_type"] = detect_event_type(text)

    return item


# =========================
# 3. BUILD EMBEDDING TEXT
# =========================
def build_text(item: Dict) -> str:
    return "\n".join([
        f"Title: {item.get('title', '')}",
        f"Summary: {item.get('summary', '')}",
        f"Published: {item.get('published_datetime', '')}",
        f"Event type: {item.get('event_type', '')}",
        f"Games: {', '.join(item.get('game_titles', []))}",
        f"Tech: {', '.join(item.get('technology_tags', []))}",
        f"Behavior: {', '.join(item.get('behavioral_tags', []))}",
    ])


# =========================
# 4. BUILD VECTOR DB
# =========================
def build_vector_db(news_items: List[Dict]):
    processed = [preprocess_news(n.copy()) for n in news_items]

    texts = [build_text(n) for n in processed]

    metadatas = []
    for n in processed:
        metadatas.append({
            "title": n.get("title", ""),
            "published_datetime": n.get("published_datetime", ""),
            "event_type": n.get("event_type", ""),
            "games": ", ".join(n.get("game_titles", [])),
        })

    embedding = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

    Chroma.from_texts(
        texts=texts,
        embedding=embedding,
        metadatas=metadatas,
        persist_directory=DB_DIR,
    )

    print("✅ Vector DB built.")


# =========================
# 5. RETRIEVER
# =========================
def get_db():
    embedding = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
    return Chroma(persist_directory=DB_DIR, embedding_function=embedding)


def parse_dt(value: str):
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except:
        return datetime.min.replace(tzinfo=timezone.utc)


def is_timeline_query(query: str) -> bool:
    q = query.lower()
    return any(x in q for x in [
        "over time", "history", "evolve", "change", "trend"
    ])


def retrieve_news(query: str, k: int = 5):
    db = get_db()

    docs = db.similarity_search(query=query, k=k * 2)

    # Timeline mode
    if is_timeline_query(query):
        docs.sort(
            key=lambda d: parse_dt(d.metadata.get("published_datetime", "")),
            reverse=False
        )
        return docs[:k]

    # Default: semantic + recency
    docs.sort(
        key=lambda d: parse_dt(d.metadata.get("published_datetime", "")),
        reverse=True
    )
    return docs[:k]


# =========================
# 6. MAIN TEST
# =========================
if __name__ == "__main__":
    filepath = "data/news.json"

    news_data = load_news_json(filepath)

    build_vector_db(news_data)

    results = retrieve_news("how has fortnite changed over time", k=5)

    for r in results:
        print("-----")
        print(r.metadata["title"])
        print(r.page_content[:200])

ModuleNotFoundError: No module named 'langchain_chroma'

In [ ]:
import os
import re
import json
import requests
import xml.etree.ElementTree as ET
from urllib.parse import urljoin, urlparse

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

FEEDS_PAGE = "https://corp.ign.com/feeds"
OUTPUT_FILE = os.path.join(DATA_DIR, "ign_feeds.json")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; IGNFeedIngestor/1.0)"
}


def fetch_page(url: str) -> str:
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    return resp.text


def find_feed_links(html: str, base_url: str) -> list[str]:
    """
    Find likely RSS/XML/JSON feed links on the IGN feeds page.
    """
    candidates = set()

    # href="..."
    hrefs = re.findall(r'href=["\'](.*?)["\']', html, flags=re.IGNORECASE)
    for href in hrefs:
        full_url = urljoin(base_url, href)

        # likely feed patterns
        if any(
            token in full_url.lower()
            for token in ["/rss/", ".xml", "feed", ".json", "feeds"]
        ):
            candidates.add(full_url)

    # also catch raw URLs present in page text
    raw_urls = re.findall(r'https?://[^\s"\'<>]+', html, flags=re.IGNORECASE)
    for raw in raw_urls:
        if any(
            token in raw.lower()
            for token in ["/rss/", ".xml", "feed", ".json", "feeds"]
        ):
            candidates.add(raw)

    return sorted(candidates)


def strip_namespace(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def get_child_text(node, names: list[str]) -> str:
    """
    Return first matching child text by local tag name.
    """
    if node is None:
        return ""
    for child in node.iter():
        if strip_namespace(child.tag) in names and child.text:
            return child.text.strip()
    return ""


def parse_rss_or_atom(xml_text: str, source_url: str) -> list[dict]:
    """
    Parse RSS or Atom feeds into a normalized JSON structure.
    """
    root = ET.fromstring(xml_text)
    items = []

    root_name = strip_namespace(root.tag).lower()

    # RSS
    if root_name == "rss":
        channel = None
        for child in root:
            if strip_namespace(child.tag).lower() == "channel":
                channel = child
                break

        if channel is None:
            return items

        for item in channel:
            if strip_namespace(item.tag).lower() != "item":
                continue

            record = {
                "source_type": "ign_rss",
                "feed_url": source_url,
                "title": get_child_text(item, ["title"]),
                "link": get_child_text(item, ["link"]),
                "summary": get_child_text(item, ["description", "summary"]),
                "published": get_child_text(item, ["pubDate", "published", "updated"]),
                "author": get_child_text(item, ["author", "creator"]),
            }
            items.append(record)

    # Atom
    elif root_name == "feed":
        for entry in root:
            if strip_namespace(entry.tag).lower() != "entry":
                continue

            link = ""
            for child in entry:
                if strip_namespace(child.tag).lower() == "link":
                    link = child.attrib.get("href", link)

            record = {
                "source_type": "ign_atom",
                "feed_url": source_url,
                "title": get_child_text(entry, ["title"]),
                "link": link,
                "summary": get_child_text(entry, ["summary", "content"]),
                "published": get_child_text(entry, ["published", "updated"]),
                "author": get_child_text(entry, ["name", "author"]),
            }
            items.append(record)

    return items


def fetch_and_parse_feed(feed_url: str) -> list[dict]:
    """
    Download a feed URL and parse XML.
    If the endpoint is JSON, try loading it directly.
    """
    resp = requests.get(feed_url, headers=HEADERS, timeout=20)
    resp.raise_for_status()

    content_type = resp.headers.get("Content-Type", "").lower()
    text = resp.text.strip()

    # JSON feed
    if "json" in content_type or text.startswith("{") or text.startswith("["):
        data = resp.json()
        if isinstance(data, dict):
            return [data]
        if isinstance(data, list):
            return data
        return []

    # Otherwise treat as XML feed
    return parse_rss_or_atom(text, feed_url)


def dedupe_records(records: list[dict]) -> list[dict]:
    seen = set()
    out = []
    for r in records:
        key = (r.get("title", ""), r.get("link", ""))
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out


def fetch_ign_feeds(feeds_page: str = FEEDS_PAGE) -> list[dict]:
    html = fetch_page(feeds_page)
    feed_links = find_feed_links(html, feeds_page)

    print(f"Found {len(feed_links)} possible feed links.")

    all_records = []
    for url in feed_links:
        try:
            records = fetch_and_parse_feed(url)
            if records:
                print(f"Parsed {len(records)} items from: {url}")
                all_records.extend(records)
            else:
                print(f"No items parsed from: {url}")
        except Exception as e:
            print(f"Skipping {url} due to error: {e}")

    all_records = dedupe_records(all_records)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_records, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(all_records)} records to {OUTPUT_FILE}")
    return all_records


if __name__ == "__main__":
    fetch_ign_feeds()

Found 64 possible feed links.
Parsed 307 items from: http://feeds.feedburner.com/FireteamChatIgnsDestinyPodcast
Parsed 26 items from: http://feeds.feedburner.com/IGNHistoryAwesome
Parsed 20 items from: http://feeds.feedburner.com/IGNPS4Articles
Parsed 20 items from: http://feeds.feedburner.com/IGNPS4Reviews
Parsed 40 items from: http://feeds.feedburner.com/IGNPS4Videos
Parsed 20 items from: http://feeds.feedburner.com/IGNXboxOneArticles
Parsed 20 items from: http://feeds.feedburner.com/IGNXboxOneReviews
Parsed 40 items from: http://feeds.feedburner.com/IGNXboxOneVideos
Parsed 66 items from: http://feeds.feedburner.com/IgnUnfiltered
Parsed 6 items from: http://feeds.feedburner.com/ign-nintendo-switch-articles
No items parsed from: http://feeds.feedburner.com/ign-nintendo-switch-reviews
No items parsed from: http://feeds.feedburner.com/ign-nintendo-switch-videos
Parsed 20 items from: http://feeds.feedburner.com/ign/all
No items parsed from: http://feeds.feedburner.com/ign/comics-articles

In [ ]:
# news_preprocess_embedding_only.py

import re
from typing import List, Dict, Tuple

from langchain_openai import OpenAIEmbeddings


# -----------------------------
# TEXT CLEANING
# -----------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("&amp;", "&")
    text = text.replace("&quot;", '"')
    text = text.replace("&#39;", "'")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# -----------------------------
# TAG / LABEL CATALOGS
# pure embedding-based classification
# -----------------------------
TAG_CATALOG = {
    "event_type": {
        "technology_update": "news about gaming technology changes, AI features, VR, AR, rendering, engines, or technical improvements",
        "monetization_change": "news about monetization, battle passes, loot boxes, subscriptions, DLC, in-game purchases, or payment model changes",
        "release_update": "news about game launches, patches, expansions, updates, release timing, or rollout changes",
        "industry_event": "news about publishers, studios, mergers, layoffs, acquisitions, partnerships, or business changes in gaming",
        "community_reaction": "news about fan reactions, reviews, backlash, controversy, community response, or public sentiment around a game",
        "general_news": "general video game news that does not strongly fit technology, monetization, release, business, or community reaction categories"
    },
    "psychological_tags": {
        "stress": "news about stress, pressure, or psychological strain related to gaming",
        "emotion": "news about emotions, feelings, mood, or emotional reactions in gaming",
        "anxiety": "news about anxiety, fear, worry, or mental unease related to gaming",
        "immersion": "news about immersion, presence, deep absorption, or feeling drawn into a game",
        "motivation": "news about motivation, player drive, goals, or reasons for continued play",
        "attention": "news about attention, focus, concentration, or cognitive engagement in games",
        "mental_health": "news about mental health, well-being, emotional wellness, or psychological impact of gaming"
    },
    "behavioral_tags": {
        "habit": "news about habitual play, repeated play patterns, or routine gaming behavior",
        "engagement": "news about engagement, retention, player activity, or continued involvement in games",
        "addiction": "news about addiction, compulsion, overuse, or problematic gaming behavior",
        "competition": "news about competition, rivalry, ranked play, or competitive gaming behavior",
        "cooperation": "news about cooperation, teamwork, collaboration, or group play in gaming",
        "social_interaction": "news about communities, group play, social interaction, communication, or multiplayer relationships in games",
        "violence": "news about violence, aggression, harmful actions, or violence-related concerns in gaming"
    },
    "technology_tags": {
        "ai": "news about artificial intelligence, generative AI, machine learning, or AI-driven game features",
        "vr": "news about virtual reality, VR headsets, or immersive virtual experiences in games",
        "ar": "news about augmented reality, AR experiences, or mixed-reality game features",
        "ray_tracing": "news about ray tracing, graphics rendering, lighting technology, or visual performance in games",
        "cloud_gaming": "news about cloud gaming, game streaming, or server-based delivery of games",
        "engine": "news about game engines, engine upgrades, rendering systems, or core technical infrastructure in games",
        "procedural_generation": "news about procedural generation, algorithmic content creation, or auto-generated worlds and systems"
    },
    "monetization_tags": {
        "microtransactions": "news about microtransactions, in-game purchases, or small paid digital items in games",
        "battle_pass": "news about battle passes, season rewards, progression monetization, or pass-based engagement systems",
        "loot_boxes": "news about loot boxes, chance-based rewards, randomized monetization, or gambling-like mechanics",
        "subscription": "news about subscriptions, recurring payments, monthly access, or membership-based game services",
        "dlc": "news about downloadable content, paid expansions, add-ons, or extra purchasable content for games",
        "free_to_play": "news about free-to-play games, monetized free access, or revenue models based on free entry"
    }
}


# -----------------------------
# COSINE SIMILARITY
# -----------------------------
def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    dot = sum(a * b for a, b in zip(vec1, vec2))
    norm1 = sum(a * a for a in vec1) ** 0.5
    norm2 = sum(b * b for b in vec2) ** 0.5
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)


# -----------------------------
# BUILD LABEL EMBEDDING INDEX
# -----------------------------
def build_label_index(embedding_model: OpenAIEmbeddings) -> Dict[str, List[Tuple[str, str, List[float]]]]:
    label_index = {}

    for category, labels in TAG_CATALOG.items():
        label_names = list(labels.keys())
        descriptions = list(labels.values())
        vectors = embedding_model.embed_documents(descriptions)

        label_index[category] = []
        for name, desc, vec in zip(label_names, descriptions, vectors):
            label_index[category].append((name, desc, vec))

    return label_index


# -----------------------------
# ASSIGN ONE LABEL
# for event_type
# -----------------------------
def assign_single_label(
    text: str,
    embedding_model: OpenAIEmbeddings,
    label_entries: List[Tuple[str, str, List[float]]]
) -> str:
    text_vec = embedding_model.embed_query(text)

    scored = []
    for label_name, _, label_vec in label_entries:
        score = cosine_similarity(text_vec, label_vec)
        scored.append((label_name, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[0][0] if scored else ""


# -----------------------------
# ASSIGN MULTI LABELS
# for tag categories
# -----------------------------
def assign_multi_labels(
    text: str,
    embedding_model: OpenAIEmbeddings,
    label_entries: List[Tuple[str, str, List[float]]],
    threshold: float = 0.30,
    top_k: int = 2
) -> List[str]:
    text_vec = embedding_model.embed_query(text)

    scored = []
    for label_name, _, label_vec in label_entries:
        score = cosine_similarity(text_vec, label_vec)
        scored.append((label_name, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [label for label, score in scored[:top_k] if score >= threshold]


# -----------------------------
# BUILD RETRIEVAL TEXT
# -----------------------------
def build_retrieval_text(item: Dict) -> str:
    return "\n".join([
        f"Title: {item.get('title', '')}",
        f"Summary: {item.get('summary', '')}",
        f"Published: {item.get('published', '')}",
        f"Author: {item.get('author', '')}",
        f"Event type: {item.get('event_type', '')}",
        f"Psychological tags: {', '.join(item.get('psychological_tags', []))}",
        f"Behavioral tags: {', '.join(item.get('behavioral_tags', []))}",
        f"Technology tags: {', '.join(item.get('technology_tags', []))}",
        f"Monetization tags: {', '.join(item.get('monetization_tags', []))}",
    ])


# -----------------------------
# PREPROCESS SINGLE RECORD
# schema stays consistent with ingestion
# -----------------------------
def preprocess_record(
    item: Dict,
    embedding_model: OpenAIEmbeddings,
    label_index: Dict[str, List[Tuple[str, str, List[float]]]],
    threshold: float = 0.30,
    top_k: int = 2
) -> Dict:
    processed = dict(item)

    # keep ingestion field names unchanged
    processed["title"] = clean_text(processed.get("title", ""))
    processed["summary"] = clean_text(processed.get("summary", ""))
    processed["author"] = clean_text(processed.get("author", ""))
    processed["published"] = processed.get("published", "").strip()

    combined_text = f"{processed['title']} {processed['summary']}".strip()

    processed["event_type"] = assign_single_label(
        text=combined_text,
        embedding_model=embedding_model,
        label_entries=label_index["event_type"]
    )

    processed["psychological_tags"] = assign_multi_labels(
        text=combined_text,
        embedding_model=embedding_model,
        label_entries=label_index["psychological_tags"],
        threshold=threshold,
        top_k=top_k
    )

    processed["behavioral_tags"] = assign_multi_labels(
        text=combined_text,
        embedding_model=embedding_model,
        label_entries=label_index["behavioral_tags"],
        threshold=threshold,
        top_k=top_k
    )

    processed["technology_tags"] = assign_multi_labels(
        text=combined_text,
        embedding_model=embedding_model,
        label_entries=label_index["technology_tags"],
        threshold=threshold,
        top_k=top_k
    )

    processed["monetization_tags"] = assign_multi_labels(
        text=combined_text,
        embedding_model=embedding_model,
        label_entries=label_index["monetization_tags"],
        threshold=threshold,
        top_k=top_k
    )

    processed["retrieval_text"] = build_retrieval_text(processed)

    return processed


# -----------------------------
# PREPROCESS LIST
# input: List[dict]
# output: List[dict]
# -----------------------------
def preprocess_news(
    news_items: List[Dict],
    threshold: float = 0.30,
    top_k: int = 2
) -> List[Dict]:
    embedding_model = OpenAIEmbeddings()
    label_index = build_label_index(embedding_model)

    return [
        preprocess_record(
            item=item,
            embedding_model=embedding_model,
            label_index=label_index,
            threshold=threshold,
            top_k=top_k
        )
        for item in news_items
    ]

In [ ]:
from typing import List, Dict

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings


def build_news_vector_store(
    processed_items: List[Dict],
    persist_directory: str = "data/news_chroma_db"
) -> Chroma:
    """
    Build a Chroma vector store from preprocessed news items.
    Each item should already contain 'retrieval_text'.
    """

    texts = [item["retrieval_text"] for item in processed_items]

    metadatas = []
    for item in processed_items:
        metadatas.append({
            "source_type": item.get("source_type", ""),
            "feed_url": item.get("feed_url", ""),
            "title": item.get("title", ""),
            "link": item.get("link", ""),
            "summary": item.get("summary", ""),
            "published": item.get("published", ""),
            "author": item.get("author", ""),
            "event_type": item.get("event_type", ""),
            "psychological_tags": ", ".join(item.get("psychological_tags", [])),
            "behavioral_tags": ", ".join(item.get("behavioral_tags", [])),
            "technology_tags": ", ".join(item.get("technology_tags", [])),
            "monetization_tags": ", ".join(item.get("monetization_tags", [])),
        })

    embeddings = OpenAIEmbeddings()

    db = Chroma.from_texts(
        texts=texts,
        embedding=embeddings,
        metadatas=metadatas,
        persist_directory=persist_directory,
    )

    return db


def load_news_vector_store(
    persist_directory: str = "data/news_chroma_db"
) -> Chroma:
    """
    Reload an existing Chroma vector store.
    """
    embeddings = OpenAIEmbeddings()

    db = Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings
    )

    return db

In [ ]:
from datetime import datetime, timezone
from typing import List, Dict, Optional

from news_vector_store import load_news_vector_store


def parse_published(value: str):
    """
    Safely parse the 'published' field from your ingestion schema.
    Returns a datetime; if parsing fails, returns the earliest possible UTC datetime.
    """
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except Exception:
        return datetime.min.replace(tzinfo=timezone.utc)


def is_timeline_query(query: str) -> bool:
    """
    Detect broad user questions about change over time.
    """
    q = query.lower()
    triggers = [
        "over time",
        "changed over time",
        "how has",
        "timeline",
        "history",
        "evolved",
        "shifted",
        "trend",
        "trends",
        "recent years",
    ]
    return any(trigger in q for trigger in triggers)


def retrieve_news(
    query: str,
    k: int = 8,
    persist_directory: str = "data/news_chroma_db",
    metadata_filter: Optional[Dict] = None
):
    """
    Retrieve news records from the vector store.

    Behavior:
    - first retrieves a larger semantic candidate pool
    - if the query looks timeline-oriented, sorts by published date ascending
    - otherwise sorts by published date descending to favor newer relevant items

    Args:
        query: user query
        k: number of final results to return
        persist_directory: Chroma persistence directory
        metadata_filter: optional Chroma metadata filter

    Returns:
        list of LangChain Document objects
    """
    db = load_news_vector_store(persist_directory=persist_directory)

    # Retrieve a larger candidate set first
    candidate_k = max(k * 2, 10)

    docs = db.similarity_search(
        query=query,
        k=candidate_k,
        filter=metadata_filter
    )

    if is_timeline_query(query):
        # for "how has X changed over time", sort oldest -> newest
        docs.sort(key=lambda d: parse_published(d.metadata.get("published", "")))
    else:
        # for standard queries, keep more recent relevant items near the top
        docs.sort(
            key=lambda d: parse_published(d.metadata.get("published", "")),
            reverse=True
        )

    return docs[:k]


def retrieve_news_with_scores(
    query: str,
    k: int = 8,
    persist_directory: str = "data/news_chroma_db",
    metadata_filter: Optional[Dict] = None
):
    """
    Same idea as retrieve_news, but keeps the similarity scores from Chroma.

    Returns:
        list of dicts with:
        - document
        - similarity_score
        - published
        - title
    """
    db = load_news_vector_store(persist_directory=persist_directory)

    candidate_k = max(k * 2, 10)

    docs_with_scores = db.similarity_search_with_score(
        query=query,
        k=candidate_k,
        filter=metadata_filter
    )

    results = []
    for doc, score in docs_with_scores:
        results.append({
            "document": doc,
            "similarity_score": score,
            "published": doc.metadata.get("published", ""),
            "title": doc.metadata.get("title", ""),
        })

    if is_timeline_query(query):
        results.sort(key=lambda x: parse_published(x["published"]))
    else:
        results.sort(key=lambda x: parse_published(x["published"]), reverse=True)

    return results[:k]


if __name__ == "__main__":
    # Example 1: general query
    results = retrieve_news("recent monetization changes in games", k=5)

    for i, doc in enumerate(results, 1):
        print(f"\nResult {i}")
        print("Title:", doc.metadata.get("title", ""))
        print("Published:", doc.metadata.get("published", ""))
        print("Event type:", doc.metadata.get("event_type", ""))
        print("Technology tags:", doc.metadata.get("technology_tags", ""))
        print("Monetization tags:", doc.metadata.get("monetization_tags", ""))
        print("Snippet:", doc.page_content[:250], "...")

    # Example 2: timeline-style query
    timeline_results = retrieve_news("how has fortnite changed over time", k=5)

    for i, doc in enumerate(timeline_results, 1):
        print(f"\nTimeline Result {i}")
        print("Title:", doc.metadata.get("title", ""))
        print("Published:", doc.metadata.get("published", ""))

In [ ]:
from typing import List
from langchain_openai import ChatOpenAI


def generate_subqueries(
    query: str,
    model: str = "gpt-4o-mini"
) -> List[str]:
    """
    Break a broad user question into 2-4 retrieval-friendly subqueries.
    """
    llm = ChatOpenAI(model=model, temperature=0)

    prompt = f"""
You are a planning assistant for a video game news agent.

Break the following user question into 2 to 4 smaller search queries.
These smaller queries should help retrieve relevant news articles.

User question:
{query}

Rules:
- Return only the subqueries
- One per line
- No numbering
- No explanation
"""

    response = llm.invoke(prompt)
    lines = [line.strip() for line in response.content.split("\n") if line.strip()]
    return lines

In [ ]:
from typing import List
from langchain_openai import ChatOpenAI


def summarize_news_docs(
    docs: List,
    query: str,
    model: str = "gpt-4o-mini"
) -> str:
    """
    Summarize retrieved news documents for one query or subquery.
    """
    if not docs:
        return "No relevant news items were found."

    joined_text = "\n\n".join([
        f"Title: {doc.metadata.get('title', '')}\n"
        f"Published: {doc.metadata.get('published', '')}\n"
        f"Content: {doc.page_content}"
        for doc in docs
    ])

    llm = ChatOpenAI(model=model, temperature=0)

    prompt = f"""
You are a video game news summarizer.

User question:
{query}

Retrieved news:
{joined_text}

Write a concise summary that answers the question using only the retrieved news.
If the question is about change over time, describe the progression in time order.
Do not invent facts.
"""

    response = llm.invoke(prompt)
    return response.content.strip()

In [ ]:
from langchain_openai import ChatOpenAI


def review_answer(
    query: str,
    draft_answer: str,
    model: str = "gpt-4o-mini"
) -> str:
    """
    Review a draft answer for consistency, missing points, and clarity.
    """
    llm = ChatOpenAI(model=model, temperature=0)

    prompt = f"""
You are a critic for a video game news agent.

User question:
{query}

Draft answer:
{draft_answer}

Tasks:
1. Check for contradictions or unsupported claims
2. Check whether important retrieved themes may have been missed
3. Rewrite the answer so it is clearer and better organized

Return only the improved final answer.
"""

    response = llm.invoke(prompt)
    return response.content.strip()

In [ ]:
from typing import Union


def format_answer(answer: Union[str, dict]) -> dict:
    """
    Normalize output into a consistent dict format.
    """
    if isinstance(answer, str):
        return {
            "content": answer.strip(),
            "references": []
        }

    return {
        "content": answer.get("content", "").strip(),
        "references": answer.get("references", [])
    }

In [ ]:
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich.table import Table

console = Console()


def pretty_print(answer: dict):
    """
    Pretty print final answer and references.
    """
    content = answer.get("content", "")
    references = answer.get("references", [])

    body = Markdown(content)

    if references:
        ref_table = Table(title="References")
        ref_table.add_column("Title / Source", overflow="fold")

        for ref in references:
            ref_table.add_row(str(ref))

        console.print(Panel(body, title="News Agent Answer", border_style="blue"))
        console.print(ref_table)
    else:
        console.print(Panel(body, title="News Agent Answer", border_style="blue"))